# Music-CRS dense retriever on Kaggle

Select **GPU T4** (or P100 if offered) and turn Internet on. Use **Save & Run All** to keep output available after the session. This notebook trains on the HF train and dev-train sessions. It scores the diagnostic slice and leaves final-holdout scoring for Task 4.

Run a numbered fold per Kaggle session if needed. Attach the previous notebook output as an input and point `RESUME_FROM` at its `cache/dense_ft` folder. Keep the model and settings fixed across folds. The script skips completed checkpoints and fold rankings, and merges the out-of-fold parquet when all numbered shards exist.

After fold 4, run `FOLDS = ["full"]` to export the full-model diagnostic rankings. For an RRF evaluation, select `INFERENCE = "rrf"` and set `EXPORT_DIAGNOSTIC = True` once all numbered checkpoints are present. Use a fresh copy without an existing diagnostic parquet when switching inference strategy.

In [ ]:
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_URL = "https://github.com/Hemanth1102/BTP3.git"
BRANCH = "codex/task-3-dense-ft"
REPO_INPUT = None  # Optional attached source directory for a private repository.
RESUME_FROM = None  # Example: Path("/kaggle/input/<previous-output>/cache/dense_ft")
FOLDS = ["0"]  # One or more of "0", "1", "2", "3", "4", "full".
MODEL = "Qwen/Qwen3-Embedding-0.6B"
MICRO_BATCH = 8
ENCODE_BATCH = 64
INFERENCE = "full"
EXPORT_DIAGNOSTIC = "full" in FOLDS
EXPORT_BLIND = False
RUN_UNION_REPORTS = False
PLANNER_CACHE = []  # Optional diagnostic planner JSONL paths.
ROOT = WORK / "cache" / "dense_ft"

In [ ]:
import os
import shutil
import subprocess
import sys

def run(*args, env=None):
    subprocess.run([str(value) for value in args], cwd=WORK, env=env, check=True)

run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv")
run(sys.executable, "-m", "pip", "install", "-q", "transformers>=4.51,<5",
    "datasets", "bm25s", "peft>=0.14,<0.19", "pyarrow")

if REPO_INPUT is None:
    repo = WORK / "BTP3"
    if not repo.exists():
        run("git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, repo)
    source = repo / "music-crs-baselines"
else:
    source = Path(REPO_INPUT)
    if not (source / "recommender_v2").is_dir():
        source = source / "music-crs-baselines"
assert (source / "recommender_v2" / "dense_ft" / "train.py").is_file(), source

if RESUME_FROM:
    previous = Path(RESUME_FROM)
    assert (previous / "manifest.json").is_file(), previous
    for origin in previous.rglob("*"):
        if origin.is_file():
            target = ROOT / origin.relative_to(previous)
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(origin, target)

env = dict(os.environ)
env["PYTHONPATH"] = str(source) + os.pathsep + env.get("PYTHONPATH", "")
env["HF_HOME"] = "/kaggle/temp/huggingface"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def module(name, *args):
    run(sys.executable, "-u", "-m", name, *args, env=env)

## Train the selected fold and export its rankings

The logical batch contains 64 query–track pairs. Smaller micro-batches retain all 64 pairs for the contrastive loss. If a GPU runs out of memory, choose a lower `MICRO_BATCH` before starting a new run. One output directory represents one fixed training configuration.

In [ ]:
module("recommender_v2.dense_ft.train", "--output-dir", ROOT, "--device", "cuda",
       "--model", MODEL, "--micro-batch-size", MICRO_BATCH, "--folds", *FOLDS)

numbered = [fold for fold in FOLDS if fold != "full"]
if numbered:
    module("recommender_v2.dense_ft.encode", "--output-dir", ROOT, "--split", "oof",
           "--device", "cuda", "--batch-size", ENCODE_BATCH, "--folds", *numbered)

for flag, split in ((EXPORT_DIAGNOSTIC, "diagnostic"), (EXPORT_BLIND, "blind")):
    if flag:
        module("recommender_v2.dense_ft.encode", "--output-dir", ROOT, "--split", split,
               "--device", "cuda", "--batch-size", ENCODE_BATCH, "--inference", INFERENCE)

## Evidence for the PR

Share `manifest.json`, the fold and full-model `training.json` files, `diagnostic_metrics.json`, and `full_top100_diagnostic.parquet`. The standalone metrics use every diagnostic query, including misses. The diagnostic parquet lets the existing CPU recall report measure union recall with the same baseline retrievers and planner-cache settings. You can also run both union comparisons here once the diagnostic parquet exists.

In [ ]:
import json

for record in sorted((ROOT / "models").glob("*/*/training.json")):
    stats = json.loads(record.read_text(encoding="utf-8"))
    print({key: stats[key] for key in ("excluded_fold", "queries", "training_seconds", "gpu")})
metrics = ROOT / "diagnostic_metrics.json"
if metrics.is_file():
    print(metrics.read_text(encoding="utf-8"))

if RUN_UNION_REPORTS:
    baseline = "bm25 bge cf item_cf item_clap item_metadata artist album title anchor transition session_bm25 anchor_album item_cf_bpr_decay item_cf_bpr_max item_clap_decay item_clap_max item_metadata_decay item_metadata_max cooc_track".split()
    plans = ["--planner-cache-from", *PLANNER_CACHE] if PLANNER_CACHE else []
    for tag, models in (("task3_before", baseline), ("task3_after", baseline + ["dense_ft"])):
        module("recall_report", "--tag", tag, "--dense-ft-dir", ROOT, "--dense-ft-inference", INFERENCE,
               "--models", *models, *plans)